# 02 · Silver et gold

**Silver** : on transforme le bronze (tout en texte, brut) en une table **typée et propre**, en listant
**chaque règle de nettoyage et le nombre de lignes qu'elle retire** (exigé par le sujet).

**Gold** : à partir du silver, on calcule les **agrégats qui répondent à la question du persona**
(quelles stations et quels secteurs manquent de vélos ou de places, et à quelles heures de la semaine ?)
et on les exporte dans PostgreSQL pour le dashboard.

| Couche | Emplacement | Contenu |
|---|---|---|
| Entrée (bronze) | `s3a://velib/bronze/velib_status` | CSV bruts en Delta (notebook 01) |
| Silver | `s3a://velib/silver/velib_status` | lignes typées et nettoyées |
| Gold | `s3a://velib/gold/*` et PostgreSQL schéma `gold` | agrégats par station, commune, heure |

**Prérequis** : `docker compose up -d` (RustFS et PostgreSQL démarrés), notebook 01 exécuté, noyau `.venv`.

In [1]:
import sys
from pathlib import Path

# Racine du projet : on remonte depuis le dossier courant jusqu'à trouver docker-compose.yml
RACINE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docker-compose.yml").exists())
sys.path.insert(0, str(RACINE / "src"))
print("Racine du projet :", RACINE)

import config
from spark_session import get_spark

Racine du projet : d:\Users\HP\Downloads\projet-dataviz-SupDeVince\projet-dataviz


## 1. Charger le bronze

In [2]:
from pyspark import StorageLevel
from pyspark.sql import functions as F

spark = get_spark("02_silver_gold")
spark.conf.set("spark.sql.session.timeZone", "UTC")

bronze = spark.read.format("delta").load(f"{config.BRONZE_PATH}/velib_status")

attendues = [
    "identifiant_station", "nom_station", "station_en_fonctionnement", "capacite_de_la_station",
    "nombre_bornettes_libres", "nombre_total_velos_disponibles", "velos_mecaniques_disponibles",
    "velos_electriques_disponibles", "retour_velib_possible", "actualisation_de_la_donnee",
    "coordonnees_geographiques", "nom_communes_equipees", "code_insee_communes_equipees", "_snapshot_ts",
]
manquantes = [c for c in attendues if c not in bronze.columns]
assert not manquantes, f"Colonnes absentes du bronze : {manquantes}. Colonnes présentes : {bronze.columns}"

bronze.select(
    F.count("*").alias("lignes_bronze"),
    F.min("_snapshot_ts").alias("premier_instantane_utc"),
    F.max("_snapshot_ts").alias("dernier_instantane_utc"),
).show(truncate=False)

+-------------+----------------------+----------------------+
|lignes_bronze|premier_instantane_utc|dernier_instantane_utc|
+-------------+----------------------+----------------------+
|4849004      |2026-11-09 15:58:09   |2026-11-15 19:33:19   |
+-------------+----------------------+----------------------+



## 2. Typage

Chaque colonne texte est convertie en son vrai type. Une valeur illisible devient `NULL`
(elle sera traitée par les règles de la section 3). On renomme les colonnes en noms courts.
La colonne `station_opening_hours` est vide dans la source : on ne la garde pas.

In [3]:
# « 48.79, 2.45 » -> latitude et longitude
morceaux = F.split(F.col("coordonnees_geographiques"), r",\s*")

typed = bronze.select(
    F.trim(F.col("identifiant_station")).alias("station_id"),
    F.col("nom_station").alias("nom_station"),
    F.upper(F.trim(F.col("station_en_fonctionnement"))).alias("en_fonctionnement"),
    F.col("capacite_de_la_station").cast("int").alias("capacite"),
    F.col("nombre_bornettes_libres").cast("int").alias("bornettes_libres"),
    F.col("nombre_total_velos_disponibles").cast("int").alias("velos"),
    F.col("velos_mecaniques_disponibles").cast("int").alias("velos_meca"),
    F.col("velos_electriques_disponibles").cast("int").alias("velos_elec"),
    F.upper(F.trim(F.col("retour_velib_possible"))).alias("retour_possible"),
    F.col("actualisation_de_la_donnee").cast("timestamp").alias("actualisation_ts"),
    morceaux.getItem(0).cast("double").alias("lat"),
    morceaux.getItem(1).cast("double").alias("lon"),
    F.trim(F.col("nom_communes_equipees")).alias("commune"),
    F.col("code_insee_communes_equipees").alias("code_insee"),
    F.col("_snapshot_ts").alias("snapshot_ts"),
).persist(StorageLevel.MEMORY_AND_DISK)

typed.printSchema()

root
 |-- station_id: string (nullable = true)
 |-- nom_station: string (nullable = true)
 |-- en_fonctionnement: string (nullable = true)
 |-- capacite: integer (nullable = true)
 |-- bornettes_libres: integer (nullable = true)
 |-- velos: integer (nullable = true)
 |-- velos_meca: integer (nullable = true)
 |-- velos_elec: integer (nullable = true)
 |-- retour_possible: string (nullable = true)
 |-- actualisation_ts: timestamp (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- commune: string (nullable = true)
 |-- code_insee: string (nullable = true)
 |-- snapshot_ts: timestamp (nullable = true)



## 3. Règles de nettoyage

Chaque règle est appliquée l'une après l'autre ; on note combien de lignes elle retire.
Deux paramètres sont modifiables (relancer alors le notebook à partir de la section 2) :
le seuil d'obsolescence d'une donnée et la zone géographique.

In [4]:
SEUIL_OBSOLESCENCE_H = 6          # donnée plus ancienne que ça (par rapport à l'instantané) : jugée périmée
LAT_MIN, LAT_MAX = 48.0, 49.3     # zone approximative de l'Île-de-France
LON_MIN, LON_MAX = 1.4, 3.6

journal = []
etat = {"n": typed.count()}
n_bronze = etat["n"]
print(f"Lignes au départ (bronze) : {n_bronze:,}".replace(",", " "))


def regle(df, nom, garder, justification):
    sortie = df.filter(garder)
    n_apres = sortie.count()
    journal.append({"regle": nom, "justification": justification, "retirees": etat["n"] - n_apres})
    etat["n"] = n_apres
    return sortie


df = typed

df = regle(df, "Identifiant de station manquant",
           F.col("station_id").isNotNull() & (F.col("station_id") != ""),
           "Sans identifiant, la ligne ne peut être rattachée à aucune station.")

df = regle(df, "Compteurs manquants ou illisibles",
           F.col("capacite").isNotNull() & F.col("bornettes_libres").isNotNull() & F.col("velos").isNotNull()
           & F.col("velos_meca").isNotNull() & F.col("velos_elec").isNotNull(),
           "Capacité, places libres et vélos sont indispensables pour détecter pénuries et saturations.")

df = regle(df, "Valeurs impossibles (négatives, capacité nulle)",
           (F.col("capacite") > 0) & (F.col("bornettes_libres") >= 0) & (F.col("velos") >= 0)
           & (F.col("velos_meca") >= 0) & (F.col("velos_elec") >= 0),
           "Un nombre négatif de vélos ou de places, ou une capacité nulle, est une erreur de mesure.")

df = regle(df, "Total de vélos différent de mécaniques + électriques",
           F.col("velos") == F.col("velos_meca") + F.col("velos_elec"),
           "Le total doit être la somme des deux types ; sinon la ligne est incohérente.")

df = regle(df, "Plus de vélos ou de places que la capacité",
           (F.col("velos") <= F.col("capacite")) & (F.col("bornettes_libres") <= F.col("capacite")),
           "Incohérent avec la capacité de la station.")

df = regle(df, "Station hors service",
           F.col("en_fonctionnement") == "OUI",
           "Une station hors service n'a pas de disponibilité réelle et fausserait les taux de pénurie.")

df = regle(df, "Coordonnées manquantes ou hors Île-de-France",
           F.col("lat").between(LAT_MIN, LAT_MAX) & F.col("lon").between(LON_MIN, LON_MAX),
           "Sans coordonnées plausibles, la station ne peut pas être placée sur une carte.")

# Âge de la donnée = écart entre l'instantané et la dernière mise à jour de la station
df = df.withColumn("age_donnee_min", (F.unix_timestamp("snapshot_ts") - F.unix_timestamp("actualisation_ts")) / 60)

quantiles = df.select(
    F.expr("percentile_approx(age_donnee_min, array(0.5, 0.9, 0.99, 0.999))").alias("q")
).first()["q"]
if quantiles:
    print("Âge de la donnée en minutes : médiane %.0f | 90 %% %.0f | 99 %% %.0f | 99,9 %% %.0f" % tuple(quantiles))
else:
    print("Âge de la donnée incalculable : vérifier la colonne actualisation_de_la_donnee")

df = regle(df, f"Donnée non actualisée depuis plus de {SEUIL_OBSOLESCENCE_H} h",
           F.col("age_donnee_min").isNull() | (F.col("age_donnee_min") <= SEUIL_OBSOLESCENCE_H * 60),
           "Une station qui ne remonte plus d'information donne un état figé, pas la réalité.")

# Doublons : une station ne peut apparaître qu'une fois par instantané
avant = etat["n"]
df = df.dropDuplicates(["station_id", "snapshot_ts"])
apres = df.count()
journal.append({"regle": "Doublons (station, instantané)",
                "justification": "Une même station ne doit apparaître qu'une fois par instantané.",
                "retirees": avant - apres})
etat["n"] = apres

Lignes au départ (bronze) : 4 849 004
Âge de la donnée en minutes : médiane 37 | 90 % 62 | 99 % 180 | 99,9 % 684620


## 4. Écriture de la couche silver

In [5]:
silver = (
    df
    .withColumn("snapshot_local_ts", F.from_utc_timestamp("snapshot_ts", "Europe/Paris"))
    .withColumn("snapshot_date", F.to_date("snapshot_local_ts"))
)

SILVER = f"{config.SILVER_PATH}/velib_status"
(
    silver.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .partitionBy("snapshot_date")
    .save(SILVER)
)
typed.unpersist()
print("Écrit dans", SILVER)

Écrit dans s3a://velib/silver/velib_status


## 5. Bilan : tableau des règles

La somme des lignes retirées doit correspondre exactement à l'écart bronze → silver.
**Copier le tableau affiché dans `docs/regles_nettoyage.md`.**

In [6]:
silver_lu = spark.read.format("delta").load(SILVER)
n_silver = silver_lu.count()
total_retire = sum(r["retirees"] for r in journal)
assert n_bronze - total_retire == n_silver, "les règles n'expliquent pas tout l'écart bronze -> silver"


def fmt(n):
    return f"{n:,}".replace(",", " ")


print(f"Bronze : {fmt(n_bronze)} lignes | Silver : {fmt(n_silver)} lignes | retirées : {fmt(total_retire)}\n")
print("| # | Règle | Justification | Lignes retirées |")
print("|---|---|---|---|")
for i, r in enumerate(journal, 1):
    print(f"| {i} | {r['regle']} | {r['justification']} | {fmt(r['retirees'])} |")

Bronze : 4 849 004 lignes | Silver : 4 567 463 lignes | retirées : 281 541

| # | Règle | Justification | Lignes retirées |
|---|---|---|---|
| 1 | Identifiant de station manquant | Sans identifiant, la ligne ne peut être rattachée à aucune station. | 34 918 |
| 2 | Compteurs manquants ou illisibles | Capacité, places libres et vélos sont indispensables pour détecter pénuries et saturations. | 178 074 |
| 3 | Valeurs impossibles (négatives, capacité nulle) | Un nombre négatif de vélos ou de places, ou une capacité nulle, est une erreur de mesure. | 9 138 |
| 4 | Total de vélos différent de mécaniques + électriques | Le total doit être la somme des deux types ; sinon la ligne est incohérente. | 0 |
| 5 | Plus de vélos ou de places que la capacité | Incohérent avec la capacité de la station. | 9 025 |
| 6 | Station hors service | Une station hors service n'a pas de disponibilité réelle et fausserait les taux de pénurie. | 9 136 |
| 7 | Coordonnées manquantes ou hors Île-de-France | Sans 

## 6. Gold : les agrégats pour le persona

On travaille en **heure de Paris**. Deux indicateurs sur chaque mesure (une station à un instant) :

- **pénurie** : la station n'a plus aucun vélo (`SEUIL_PENURIE = 0`) ;
- **saturation** : la station n'a plus aucune place libre (`SEUIL_SATURATION = 0`).

Un **taux** est la part des mesures où l'événement se produit (par exemple 0,12 = 12 % du temps).

| Table gold | Une ligne par… | Sert à |
|---|---|---|
| `station_heure` | station × type de jour × heure | carte et classement des stations, filtre par heure |
| `commune_heure` | commune × type de jour × heure | comparer les secteurs |
| `reseau_heure` | jour de la semaine × heure | profil hebdomadaire de tout le réseau |
| `station_resume` | station | classement global, KPI |

**Stations mortes** : une station sans vélo plus de 99 % du temps (`SEUIL_MORTE`) n'est pas « en tension », elle est
probablement en panne alors qu'elle est déclarée en fonctionnement. On les repère avec `station_morte = true`.
Elles restent dans `station_heure` et `station_resume` (pour pouvoir les citer), mais sont **exclues** de
`commune_heure` et `reseau_heure`, pour que les moyennes reflètent le fonctionnement réel du réseau.

In [7]:
SEUIL_PENURIE = 0        # pénurie : au plus ce nombre de vélos
SEUIL_SATURATION = 0     # saturation : au plus ce nombre de places libres
SEUIL_MORTE = 0.99       # station « morte » : taux de pénurie sur toute la période au moins égal à ça

s = spark.read.format("delta").load(SILVER)

base = (
    s.withColumn("heure", F.hour("snapshot_local_ts"))
     .withColumn("jour_semaine", (F.dayofweek("snapshot_local_ts") + 5) % 7 + 1)   # 1 = lundi ... 7 = dimanche
     .withColumn("jour_type", F.when(F.col("jour_semaine") >= 6, "week-end").otherwise("semaine"))
     .withColumn("penurie", (F.col("velos") <= SEUIL_PENURIE).cast("int"))
     .withColumn("saturation", (F.col("bornettes_libres") <= SEUIL_SATURATION).cast("int"))
     .withColumn("remplissage", F.col("velos") / F.col("capacite"))
     .withColumn("penurie_elec", (F.col("velos_elec") == 0).cast("int"))
     .withColumn("penurie_meca", (F.col("velos_meca") == 0).cast("int"))
)

# Stations mortes : repérées sur toute la période
mortes = (
    base.groupBy("station_id").agg(F.avg("penurie").alias("tp"))
        .select("station_id", (F.col("tp") >= SEUIL_MORTE).alias("station_morte"))
)
base = base.join(mortes, "station_id").persist(StorageLevel.MEMORY_AND_DISK)
vivante = base.filter(~F.col("station_morte"))

nb_mortes = mortes.filter("station_morte").count()
print(f"Stations mortes (sans vélo plus de {SEUIL_MORTE:.0%} du temps) : {nb_mortes} sur {mortes.count()}")
(
    base.filter("station_morte").groupBy("station_id")
        .agg(F.max("nom_station").alias("nom_station"), F.max("commune").alias("commune"),
             F.avg("penurie").alias("taux_penurie"), F.avg("saturation").alias("taux_saturation"))
        .orderBy("station_id").show(50, truncate=False)
)

taux = [
    F.count("*").alias("nb_mesures"),
    F.avg("penurie").alias("taux_penurie"),
    F.avg("saturation").alias("taux_saturation"),
    F.avg("remplissage").alias("remplissage_moyen"),
]

station_heure = base.groupBy("station_id", "jour_type", "heure").agg(
    F.max("nom_station").alias("nom_station"),
    F.max("commune").alias("commune"),
    F.avg("lat").alias("lat"),
    F.avg("lon").alias("lon"),
    F.max("capacite").alias("capacite"),
    *taux,
    F.avg("velos").alias("velos_moyens"),
    F.avg("penurie_elec").alias("taux_penurie_elec"),
    F.avg("penurie_meca").alias("taux_penurie_meca"),
    F.first("station_morte").alias("station_morte"),
)

commune_heure = vivante.groupBy("commune", "jour_type", "heure").agg(
    F.countDistinct("station_id").alias("nb_stations"),
    *taux,
)

reseau_heure = vivante.groupBy("jour_semaine", "heure").agg(
    *taux,
    (F.sum("velos_elec") / F.sum("velos")).alias("part_velos_electriques"),
)

station_resume = base.groupBy("station_id").agg(
    F.max("nom_station").alias("nom_station"),
    F.max("commune").alias("commune"),
    F.avg("lat").alias("lat"),
    F.avg("lon").alias("lon"),
    F.max("capacite").alias("capacite"),
    *taux,
    F.first("station_morte").alias("station_morte"),
)


def arrondir(d):
    for nom, type_ in d.dtypes:
        if type_ == "double":
            d = d.withColumn(nom, F.round(F.col(nom), 6 if nom in ("lat", "lon") else 4))
    return d


gold = {
    "station_heure": arrondir(station_heure),
    "commune_heure": arrondir(commune_heure),
    "reseau_heure": arrondir(reseau_heure),
    "station_resume": arrondir(station_resume),
}

for nom, d in gold.items():
    (
        d.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", True)
        .save(f"{config.GOLD_PATH}/{nom}")
    )
    print(f"{nom:15} {fmt(spark.read.format('delta').load(config.GOLD_PATH + '/' + nom).count()):>10} lignes")
base.unpersist()

Stations mortes (sans vélo plus de 99% du temps) : 6 sur 1504
+----------+-------------------------------+-----------+------------------+---------------+
|station_id|nom_station                    |commune    |taux_penurie      |taux_saturation|
+----------+-------------------------------+-----------+------------------+---------------+
|3006      |Mairie du 3ème                 |Paris      |1.0               |0.0            |
|32305     |Saint-Germain - Verdun         |Romainville|1.0               |0.0            |
|41104     |Vaillant Couturier - Desmoulins|Arcueil    |0.9993434011818779|1.0            |
|5110      |Lacépède - Monge               |Paris      |1.0               |0.0            |
|7005      |Raspail - Bac                  |Paris      |0.9908076165462902|0.0            |
|9010      |Rochechouart lamartine         |Paris      |1.0               |0.0            |
+----------+-------------------------------+-----------+------------------+---------------+

station_heure    

DataFrame[station_id: string, nom_station: string, en_fonctionnement: string, capacite: int, bornettes_libres: int, velos: int, velos_meca: int, velos_elec: int, retour_possible: string, actualisation_ts: timestamp, lat: double, lon: double, commune: string, code_insee: string, snapshot_ts: timestamp, age_donnee_min: double, snapshot_local_ts: timestamp, snapshot_date: date, heure: int, jour_semaine: int, jour_type: string, penurie: int, saturation: int, remplissage: double, penurie_elec: int, penurie_meca: int, station_morte: boolean]

## 7. Export vers PostgreSQL

Chaque table gold est copiée dans le schéma `gold` de PostgreSQL (`gold.station_heure`, etc.).
C'est PostgreSQL que le dashboard Streamlit interroge. Relancer la cellule remplace le contenu.

In [8]:
import psycopg2

with psycopg2.connect(
    host=config.PG_HOST, port=config.PG_PORT, dbname=config.PG_DB,
    user=config.PG_USER, password=config.PG_PASSWORD,
) as conn:
    with conn.cursor() as cur:
        cur.execute("CREATE SCHEMA IF NOT EXISTS gold")
        # On supprime les anciennes tables : leur structure peut avoir changé (nouvelles colonnes)
        for nom in gold:
            cur.execute(f"DROP TABLE IF EXISTS gold.{nom}")


def vers_postgres(d, table):
    (
        d.write.format("jdbc")
        .option("url", config.PG_JDBC_URL)
        .option("dbtable", table)
        .option("user", config.PG_USER)
        .option("password", config.PG_PASSWORD)
        .option("driver", "org.postgresql.Driver")
        .mode("overwrite")
        .save()
    )


for nom, d in gold.items():
    vers_postgres(d, f"gold.{nom}")

with psycopg2.connect(
    host=config.PG_HOST, port=config.PG_PORT, dbname=config.PG_DB,
    user=config.PG_USER, password=config.PG_PASSWORD,
) as conn:
    with conn.cursor() as cur:
        for nom in gold:
            cur.execute(f"SELECT count(*) FROM gold.{nom}")
            print(f"gold.{nom:15} {fmt(cur.fetchone()[0]):>10} lignes dans PostgreSQL")

gold.station_heure       46 584 lignes dans PostgreSQL
gold.commune_heure        2 139 lignes dans PostgreSQL
gold.reseau_heure           105 lignes dans PostgreSQL
gold.station_resume       1 504 lignes dans PostgreSQL


## 8. Premier coup d'œil aux résultats

Pour vérifier que les agrégats ont du sens avant de construire le dashboard.

In [9]:
print("Les 10 stations les plus souvent sans vélo (stations mortes exclues, au moins 200 mesures) :")
(
    gold["station_resume"]
    .filter((F.col("nb_mesures") >= 200) & ~F.col("station_morte"))
    .orderBy(F.desc("taux_penurie"))
    .select("station_id", "nom_station", "commune", "capacite", "nb_mesures", "taux_penurie", "taux_saturation")
    .show(10, truncate=False)
)

print("Profil du réseau par heure (lundi à vendredi) :")
(
    gold["reseau_heure"]
    .filter(F.col("jour_semaine") <= 5)
    .groupBy("heure")
    .agg(F.avg("taux_penurie").alias("taux_penurie"), F.avg("taux_saturation").alias("taux_saturation"))
    .orderBy("heure")
    .show(24)
)

Les 10 stations les plus souvent sans vélo (stations mortes exclues, au moins 200 mesures) :


+----------+----------------------------------+-------+--------+----------+------------+---------------+
|station_id|nom_station                       |commune|capacite|nb_mesures|taux_penurie|taux_saturation|
+----------+----------------------------------+-------+--------+----------+------------+---------------+
|8201      |Arsène Houssaye - Friedland       |Paris  |29      |3046      |0.4852      |0.0            |
|8057      |Hoche - Tilsitt                   |Paris  |33      |3046      |0.4708      |0.0            |
|23106     |Pavé des Gardes - Charles Vaillant|Sèvres |20      |2973      |0.4598      |0.0            |
|8056      |Beaujon - Wagram                  |Paris  |29      |3046      |0.4583      |0.0            |
|8045      |Place de la Reine Astrid          |Paris  |58      |3046      |0.4531      |0.0033         |
|8029      |Place du Canada                   |Paris  |58      |3046      |0.4301      |0.0788         |
|8001      |Petit Palais                      |Paris  |

In [10]:
from pyspark.sql import Window

snaps = spark.read.format("delta").load(f"{config.BRONZE_PATH}/velib_status").select("_snapshot_ts").distinct()
w = Window.orderBy("_snapshot_ts")
trous = (
    snaps.withColumn("precedent", F.lag("_snapshot_ts").over(w))
    .withColumn("minutes", F.round((F.unix_timestamp("_snapshot_ts") - F.unix_timestamp("precedent")) / 60))
    .filter("minutes > 10")
    .select(
        F.from_utc_timestamp("precedent", "Europe/Paris").alias("debut_trou"),
        F.from_utc_timestamp("_snapshot_ts", "Europe/Paris").alias("fin_trou"),
        "minutes",
    )
    .orderBy("debut_trou")
)
trous.show(50, truncate=False)

+-------------------+-------------------+-------+
|debut_trou         |fin_trou           |minutes|
+-------------------+-------------------+-------+
|2026-11-10 11:24:36|2026-11-10 12:27:21|63.0   |
|2026-11-11 04:24:36|2026-11-11 05:56:38|92.0   |
|2026-11-11 12:38:36|2026-11-11 12:49:49|11.0   |
|2026-11-13 09:15:49|2026-11-13 09:50:03|34.0   |
|2026-11-13 19:35:49|2026-11-14 10:05:08|869.0  |
|2026-11-14 15:45:50|2026-11-15 20:31:19|1725.0 |
+-------------------+-------------------+-------+



## À retenir pour le README et les fiches KPI

- Lignes bronze, lignes silver, lignes retirées par règle : *tableau de la section 5*.
- Période couverte et nombre de stations : *à noter ici*.
- Premières observations (heures et stations en tension) : *à noter ici*.